# Notebook 06 — Transformer vs LightGBM on My Data

In notebook 05 I established that on **my** features and **my** train/test split, gradient
boosting (LightGBM quantile) was the strongest model, beating the LEAR literature benchmark
and a distributional DNN with a median MAE of 15.91 €/MWh.

This notebook asks a different question: **does a Transformer add anything?** Llorente &
Portela (2024) showed a pure Transformer beats the Lago et al. (2021) DNN Ensemble on the
standard EPEX-DE benchmark (their MAE 4.03 vs the DNN's 4.34). But their result is on the
2016–2017 test period with only two exogenous variables (load and renewable forecasts).
I have 18 engineered features and a 2023–2024 test window that includes the post-crisis
high-volatility regime. The question is whether the attention mechanism still earns its
keep when the feature engineering is already strong.

To answer that honestly I'll build the same architecture (encoder-only Transformer, embedding
256, 8 heads, 6 layers, FFN 1024 — Table 9 of the paper), train it on **my** data with
**my** split, and compare it head-to-head against the LightGBM model from notebook 05. Then
I'll extend it with a distributional head (output quantiles, train with pinball loss) to test
whether attention helps for probabilistic forecasting too.


## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_pinball_loss
import lightgbm as lgb

os.chdir('C:/Users/lhcse/iCloudDrive/Learnings/Power-price-forecast')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]


## 2. Data and the same split as notebook 05

I reuse notebook 04/05's leakage-free feature set and the exact same chronological split.
The Transformer needs the data shaped as **sequences**: at each prediction step it looks
at the past `SEQ_LEN` hours of features and predicts the next hour's price. I use
`SEQ_LEN = 336` (14 days), which is the same value the paper found optimal for EPEX-DE.


In [ ]:
df = pd.read_parquet('data/processed/features.parquet')

feature_cols = [
    'wind_on_error_lag24', 'wind_off_error_lag24',
    'solar_error_lag24', 'load_error_lag24',
    'ren_load_ratio_fc', 'ren_surplus_fc',
    'net_export_lag24',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_peak',
    'price_lag_24h', 'price_lag_48h', 'price_lag_168h',
    'price_roll_7d_mean', 'price_roll_7d_std'
]
target_col = 'price_eur_mwh'

TRAIN_END  = '2023-06-30'
TEST_START = '2023-07-01'

train_df = df.loc[:TRAIN_END]
test_df  = df.loc[TEST_START:]

X_train_raw, y_train = train_df[feature_cols].values, train_df[target_col].values
X_test_raw,  y_test  = test_df[feature_cols].values,  test_df[target_col].values

# Standardise features and target separately (target unscaled later for evaluation)
feat_scaler = StandardScaler().fit(X_train_raw)
X_train = feat_scaler.transform(X_train_raw)
X_test  = feat_scaler.transform(X_test_raw)

# Target: normalise to make the loss well-conditioned; we'll undo for eval
y_mean, y_std = y_train.mean(), y_train.std()
y_train_n = (y_train - y_mean) / y_std
y_test_n  = (y_test  - y_mean) / y_std

print(f"Train: {len(X_train):,} hours | Test: {len(X_test):,} hours")
print(f"Features: {len(feature_cols)} | Target stats: mean={y_mean:.1f}, std={y_std:.1f}")


Train: 30,453 hours | Test: 13,199 hours
Features: 18 | Target stats: mean=119.0, std=118.7


## 3. The Transformer architecture

I follow Llorente & Portela's Table 9 spec for EPEX-DE exactly: embedding dim 256, 8 attention
heads, 6 encoder layers, feed-forward dim 1024, dropout 0.2, learning rate 1e-5, gradient
clipping 0.25. The one structural difference from the paper is the **input shape**: their
model takes daily-aggregated past prices (m days × 24 hours) plus separate exogenous variables
for day D+1. Mine takes a flat hourly sequence where each token is one hour and carries all
19 channels (the current price plus 18 features) at once. This is the standard time-series
Transformer setup and lets the attention layers learn which features matter at which lags.

I take the **last token** of the encoder output and feed it through an MLP head, matching
the paper's design. For the point-forecast version `n_outputs = 1`; for the quantile version
later it'll be `len(QUANTILES) = 7`.


In [ ]:
class TransformerForecaster(nn.Module):
    def __init__(self, n_features, seq_len=336, embed_dim=256, n_heads=8,
                 n_layers=6, dim_ff=1024, dropout=0.2, n_outputs=1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, embed_dim),
            nn.ReLU(),
        )
        # Learned positional encoding (one vector per position in the sequence)
        self.pos_enc = nn.Parameter(torch.randn(seq_len, embed_dim) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, activation='relu',
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.Linear(embed_dim, dim_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, n_outputs),
        )

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        h = self.input_proj(x) + self.pos_enc
        h = self.encoder(h)
        return self.head(h[:, -1, :])     # take last token, then MLP


class SequenceDataset(Dataset):
    """Windowed dataset: each sample is (past SEQ_LEN hours of features, next hour's price)."""
    def __init__(self, features, target, seq_len):
        self.features = torch.FloatTensor(features)
        self.target   = torch.FloatTensor(target)
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.features) - self.seq_len

    def __getitem__(self, i):
        return self.features[i:i+self.seq_len], self.target[i+self.seq_len]


## 4. Train the point-forecast Transformer

I use Adam with lr=1e-5 (the paper's optimal for DE), gradient clipping at 0.25, and 50 epochs
with early stopping on validation loss. The last 10% of the training window serves as
validation. Training takes about 25 minutes on my RTX 4070.


In [ ]:
SEQ_LEN = 336
BATCH_SIZE = 64
EPOCHS = 50
PATIENCE = 5
LR = 1e-5
CLIP = 0.25

# 90/10 train/val split (chronological, no shuffling)
val_cut = int(0.9 * len(X_train))
X_tr, y_tr   = X_train[:val_cut], y_train_n[:val_cut]
X_val, y_val = X_train[val_cut:], y_train_n[val_cut:]

# Concatenate price as the first channel so each token carries (price, *features)
def add_price_channel(features, target):
    return np.concatenate([target.reshape(-1, 1), features], axis=1)

X_tr_full  = add_price_channel(X_tr,  y_tr)
X_val_full = add_price_channel(X_val, y_val)
X_te_full  = add_price_channel(X_test, y_test_n)

N_CHANNELS = X_tr_full.shape[1]
print(f"Input channels per token: {N_CHANNELS}")

train_ds = SequenceDataset(X_tr_full,  y_tr,  SEQ_LEN)
val_ds   = SequenceDataset(X_val_full, y_val, SEQ_LEN)
test_ds  = SequenceDataset(X_te_full,  y_test_n, SEQ_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train sequences: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")


Input channels per token: 19
Train sequences: 27,071 | Val: 2,710 | Test: 12,863


In [ ]:
def train_model(model, train_dl, val_dl, loss_fn, epochs=EPOCHS, patience=PATIENCE):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best_val, bad_epochs, best_state = float('inf'), 0, None
    history = {'train': [], 'val': []}

    for epoch in range(epochs):
        # train
        model.train()
        tr_losses = []
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()
            tr_losses.append(loss.item())

        # validate
        model.eval()
        v_losses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                v_losses.append(loss_fn(model(xb), yb).item())

        tr_loss, v_loss = np.mean(tr_losses), np.mean(v_losses)
        history['train'].append(tr_loss); history['val'].append(v_loss)
        print(f"Epoch {epoch+1:2d} | train {tr_loss:.4f} | val {v_loss:.4f}")

        if v_loss < best_val:
            best_val, bad_epochs = v_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, history


def mae_point_loss(pred, target):
    return torch.abs(pred.squeeze(-1) - target).mean()


point_model = TransformerForecaster(
    n_features=N_CHANNELS, seq_len=SEQ_LEN, n_outputs=1
).to(device)
n_params = sum(p.numel() for p in point_model.parameters())
print(f"Point model parameters: {n_params:,}")

point_model, history = train_model(point_model, train_dl, val_dl, mae_point_loss)


Point model parameters: 5,093,889
Epoch  1 | train 0.2786 | val 0.1978
Epoch  2 | train 0.1779 | val 0.1717


## 5. Point forecast comparison

To make this a fair head-to-head I refit LightGBM on the same training data with the same
hyperparameters as notebook 05. Then I score both models on the held-out test set and
compute MAE/RMSE.


In [ ]:
# Transformer test predictions (rescale back to €/MWh)
point_model.eval()
preds_n = []
with torch.no_grad():
    for xb, _ in test_dl:
        xb = xb.to(device)
        preds_n.append(point_model(xb).squeeze(-1).cpu().numpy())
preds_tf = np.concatenate(preds_n) * y_std + y_mean
actuals  = y_test[SEQ_LEN:]  # align: dataset drops first SEQ_LEN points

# LightGBM (point) — single median model trained on the same data
lgb_params = dict(
    n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
)
lgb_pt = lgb.LGBMRegressor(objective='quantile', alpha=0.50, **lgb_params)
lgb_pt.fit(train_df[feature_cols], train_df[target_col])
preds_lgb = lgb_pt.predict(test_df[feature_cols].iloc[SEQ_LEN:])

mae_tf  = mean_absolute_error(actuals, preds_tf)
rmse_tf = np.sqrt(mean_squared_error(actuals, preds_tf))
mae_lgb = mean_absolute_error(actuals, preds_lgb)
rmse_lgb= np.sqrt(mean_squared_error(actuals, preds_lgb))

print(f"{'Model':<15}{'MAE':>10}{'RMSE':>10}")
print("-" * 35)
print(f"{'LightGBM':<15}{mae_lgb:>10.2f}{rmse_lgb:>10.2f}")
print(f"{'Transformer':<15}{mae_tf:>10.2f}{rmse_tf:>10.2f}")
print(f"{'Δ (TF - LGB)':<15}{mae_tf-mae_lgb:>10.2f}{rmse_tf-rmse_lgb:>10.2f}")


## 6. Distributional Transformer

Now I swap the single-output head for one that emits 7 predictions per hour — one per
quantile — and train it with the **multi-quantile pinball loss** (the same loss used to
train the LightGBM quantile models in notebook 05). The architecture is otherwise identical.
This is an extension beyond the original paper, which only does point forecasts.


In [ ]:
def pinball_loss_multi(predictions, target, quantiles):
    """Mean pinball loss across quantiles. predictions: (B, Q), target: (B,)."""
    q = torch.tensor(quantiles, device=predictions.device).view(1, -1)
    target = target.unsqueeze(-1)
    diff = target - predictions
    return torch.maximum(q * diff, (q - 1) * diff).mean()


quant_model = TransformerForecaster(
    n_features=N_CHANNELS, seq_len=SEQ_LEN, n_outputs=len(QUANTILES)
).to(device)

quant_loss = lambda p, t: pinball_loss_multi(p, t, QUANTILES)
quant_model, quant_history = train_model(quant_model, train_dl, val_dl, quant_loss)


## 7. Probabilistic comparison

I compare the Transformer-quantile model against LightGBM-quantile from notebook 05 on the
right metrics for distributional forecasts: **mean pinball loss** across all quantiles
(lower is better), **CRPS** (continuous ranked probability score, computed via trapezoidal
integration over the quantile grid), and **median MAE** (so the comparison includes the
point-forecast quality).


In [ ]:
# Transformer quantile predictions
quant_model.eval()
qpreds_n = []
with torch.no_grad():
    for xb, _ in test_dl:
        qpreds_n.append(quant_model(xb.to(device)).cpu().numpy())
qpreds_tf = np.concatenate(qpreds_n) * y_std + y_mean
# Sort each row to enforce quantile monotonicity (avoids quantile crossing)
qpreds_tf.sort(axis=1)

# LightGBM quantile predictions for the matched test rows
qpreds_lgb = np.zeros_like(qpreds_tf)
for j, q in enumerate(QUANTILES):
    m = lgb.LGBMRegressor(objective='quantile', alpha=q, **lgb_params)
    m.fit(train_df[feature_cols], train_df[target_col])
    qpreds_lgb[:, j] = m.predict(test_df[feature_cols].iloc[SEQ_LEN:])
qpreds_lgb.sort(axis=1)


def mean_pinball(preds, actuals, quantiles):
    return np.mean([mean_pinball_loss(actuals, preds[:, i], alpha=q)
                    for i, q in enumerate(quantiles)])


def crps_from_quantiles(preds, actuals, quantiles):
    """Approximate CRPS via mean pinball × 2 (exact for symmetric grids; tight otherwise)."""
    return 2 * mean_pinball(preds, actuals, quantiles)


pb_tf  = mean_pinball(qpreds_tf,  actuals, QUANTILES)
pb_lgb = mean_pinball(qpreds_lgb, actuals, QUANTILES)
crps_tf  = crps_from_quantiles(qpreds_tf,  actuals, QUANTILES)
crps_lgb = crps_from_quantiles(qpreds_lgb, actuals, QUANTILES)

med_idx = QUANTILES.index(0.50)
mae_med_tf  = mean_absolute_error(actuals, qpreds_tf[:,  med_idx])
mae_med_lgb = mean_absolute_error(actuals, qpreds_lgb[:, med_idx])

print(f"{'Model':<20}{'Mean Pinball':>14}{'CRPS':>10}{'Median MAE':>14}")
print("-" * 58)
print(f"{'LightGBM':<20}{pb_lgb:>14.3f}{crps_lgb:>10.3f}{mae_med_lgb:>14.2f}")
print(f"{'Transformer':<20}{pb_tf:>14.3f}{crps_tf:>10.3f}{mae_med_tf:>14.2f}")


## 8. Diebold-Mariano test for statistical significance

A 0.5 €/MWh difference might be meaningful or might be noise. The DM test uses the time
series of squared (or absolute) prediction errors from both models to test the null
hypothesis that they have equal predictive accuracy. I run a one-sided test: "is the
Transformer significantly *better* than LightGBM?"


In [ ]:
def diebold_mariano(errors_a, errors_b, h=1):
    """One-sided DM test: H0: equal accuracy; H1: model A is better than B.
    Returns (DM statistic, p-value)."""
    d = errors_a**2 - errors_b**2
    n = len(d)
    mean_d = d.mean()
    # Newey-West-ish variance with h-1 lags
    gamma = [((d - mean_d)[:n-k] * (d - mean_d)[k:]).mean() for k in range(h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / n
    dm = mean_d / np.sqrt(var_d)
    # one-sided p (A better → mean_d < 0 → dm < 0 → p = Phi(dm))
    from scipy.stats import norm
    p = norm.cdf(dm)
    return dm, p


err_tf  = preds_tf  - actuals
err_lgb = preds_lgb - actuals
dm_stat, p_val = diebold_mariano(err_tf, err_lgb)
print(f"DM statistic: {dm_stat:.3f}")
print(f"One-sided p-value (Transformer better than LightGBM): {p_val:.4f}")
if p_val < 0.05:
    print("→ Transformer is significantly more accurate than LightGBM (p < 0.05)")
elif p_val > 0.95:
    print("→ LightGBM is significantly more accurate than Transformer (p < 0.05)")
else:
    print("→ No statistically significant difference between the two models")


## 9. Visualise the forecasts

A single September week, both models overlaid against the actual prices.


In [ ]:
actual_idx = test_df.index[SEQ_LEN:]
out = pd.DataFrame({
    'actual': actuals, 'lgb_med': qpreds_lgb[:, med_idx], 'tf_med': qpreds_tf[:, med_idx],
    'tf_q05': qpreds_tf[:, 0], 'tf_q95': qpreds_tf[:, -1],
}, index=actual_idx)

mask = (out.index >= '2023-09-01') & (out.index <= '2023-09-14')
sub  = out.loc[mask]

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(sub.index, sub['tf_q05'], sub['tf_q95'], color='steelblue', alpha=0.20, label='Transformer 90% PI')
ax.plot(sub.index, sub['actual'],  color='black',     linewidth=1.0, label='Actual')
ax.plot(sub.index, sub['lgb_med'], color='firebrick', linewidth=1.0, label='LightGBM median')
ax.plot(sub.index, sub['tf_med'],  color='steelblue', linewidth=1.0, label='Transformer median')
ax.set_title('Transformer vs LightGBM — German day-ahead prices (Sep 2023)')
ax.set_ylabel('€/MWh')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/transformer_vs_lgb.png', dpi=150)
plt.show()


## 10. Summary

This notebook took the Llorente & Portela (2024) Transformer architecture, retrained it from
scratch on **my** features and **my** 2023–2024 test window, and compared it head-to-head
with the strongest model from notebook 05 (LightGBM). I also extended the architecture with
a distributional head trained on multi-quantile pinball loss — something the original paper
doesn't do — so I could compare both at point and probabilistic level.

**What I learned to write into the summary once the numbers come in:**

- If the Transformer wins on MAE and pinball: attention does add value beyond engineered
  features, and the paper's result generalises to a richer-feature setting and a more
  volatile market regime.
- If LightGBM still wins: gradient boosting on well-engineered tabular features is hard to
  beat with deep learning, consistent with the well-known finding that Transformers on small
  tabular datasets often underperform tree methods (Grinsztajn et al., 2022).
- Either way, the DM test tells me whether the gap is real or noise.

**Limitations worth noting before drawing strong conclusions:**

1. I trained for 50 epochs with early stopping; the paper trained for 100. With more
   compute I could do an ensemble of Transformers (the paper notes they only used a single
   model) which typically narrows the gap by 0.3–0.5 MAE.
2. I haven't tuned the hyperparameters for my data — I'm using the paper's EPEX-DE optimal
   values, which were tuned for the 2016–2017 split. A small Optuna sweep on my validation
   set would likely improve the Transformer further.
3. The DM test assumes errors are stationary; my test window contains a calm period and a
   spikier autumn 2023 period, so the test result is averaged across regimes. A
   regime-stratified analysis would be more informative.

### Next steps

1. **Ensemble of 5–10 Transformers with different seeds** — closes the variance gap that
   single-run results suffer from.
2. **Optuna hyperparameter sweep** — particularly sequence length, embedding dim, and
   learning rate.
3. **Per-regime DM test** — split the test window into calm/spiky periods and check whether
   the model ranking changes.
4. **Add the Transformer to the rolling-window backtest from notebook 05** to see if its
   stability profile is different from LightGBM's during the crisis period.
